In [1]:
from pathlib import Path
import json
import re
import warnings
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Libraries imported successfully!")


Libraries imported successfully!


In [2]:
# Find the project folder whether Jupyter was opened from the project root or notebooks/.
current_folder = Path.cwd().resolve()
if (current_folder / "data").exists():
    PROJECT_ROOT = current_folder
elif (current_folder.parent / "data").exists():
    PROJECT_ROOT = current_folder.parent
else:
    raise FileNotFoundError(
        "Could not find the SkillBridge project folder. Open this notebook from the project or notebooks folder."
    )

JOB_OUTPUT = PROJECT_ROOT / "data" / "processed" / "job_analyzer"
JOB_OUTPUT.mkdir(parents=True, exist_ok=True)

CV_OUTPUT = PROJECT_ROOT / "data" / "processed" / "cv_builder"
RESUME_LIBRARY = CV_OUTPUT / "resume_library"

MODEL_PATH = PROJECT_ROOT / "models"

# Leave as None to use the newest saved original resume.
# To choose another resume, enter its candidate_id / filename stem here.
SELECTED_RESUME_ID = None

# Semantic matching is optional. Exact skill matching works without downloading a model.
ENABLE_SEMANTIC_MATCHING = False

print("Project Root:", PROJECT_ROOT)
print("Job Output:", JOB_OUTPUT)
print("Resume Library:", RESUME_LIBRARY)
print("Model Path:", MODEL_PATH)
print("Day 10 environment ready!")


Project Root: /Users/rizwansalim/Desktop/SkillBridgeByZequa
Job Output: /Users/rizwansalim/Desktop/SkillBridgeByZequa/data/processed/job_analyzer
Resume Library: /Users/rizwansalim/Desktop/SkillBridgeByZequa/data/processed/cv_builder/resume_library
Model Path: /Users/rizwansalim/Desktop/SkillBridgeByZequa/models
Day 10 environment ready!


In [3]:
# Load the selected original resume from Notebook 09's Resume Library.
# If the library is empty, try the older single-file profile used by the first version of Notebook 09.

resume_files = []
if RESUME_LIBRARY.exists():
    resume_files = sorted(
        [path for path in RESUME_LIBRARY.glob("*.json") if path.name != "resume_library_index.json"],
        key=lambda path: path.stat().st_mtime,
        reverse=True
    )

candidate_path = None
if SELECTED_RESUME_ID:
    for path in resume_files:
        if path.stem == SELECTED_RESUME_ID:
            candidate_path = path
            break
    if candidate_path is None:
        raise FileNotFoundError(
            f"Resume ID '{SELECTED_RESUME_ID}' was not found in {RESUME_LIBRARY}. "
            "Check the candidate_id shown in Notebook 09."
        )
elif resume_files:
    candidate_path = resume_files[0]
else:
    legacy_path = CV_OUTPUT / "updated_candidate.json"
    if legacy_path.is_file():
        candidate_path = legacy_path

if candidate_path is None:
    raise FileNotFoundError(
        "No saved resume was found. Run Notebook 08 and Notebook 09 first. "
        f"Expected an original profile in: {RESUME_LIBRARY}"
    )

with candidate_path.open("r", encoding="utf-8") as file:
    candidate_data = json.load(file)

# Notebook 09 uses personal_details; this fallback also supports the raw profile format from Notebook 08.
if "personal_details" not in candidate_data:
    candidate_data["personal_details"] = {
        "full_name": candidate_data.get("name", ""),
        "email": candidate_data.get("email", ""),
        "phone": candidate_data.get("phone", ""),
        "linkedin": candidate_data.get("linkedin", ""),
        "portfolio": candidate_data.get("github", ""),
        "professional_summary": candidate_data.get("summary", "")
    }

candidate_skills = candidate_data.get("skills", [])
if isinstance(candidate_skills, str):
    candidate_skills = [item.strip() for item in candidate_skills.split(",") if item.strip()]

candidate_details = candidate_data.get("personal_details", {})
candidate_name = candidate_details.get("full_name") or candidate_data.get("name") or "Candidate"

print("Loaded resume:", candidate_path.name)
print("Candidate:", candidate_name)
print("Candidate Skills:", candidate_skills)
print("Total Skills:", len(candidate_skills))


Loaded resume: candidate-sample-resume-2.json
Candidate: Candidate
Candidate Skills: ['computer science', 'deep learning', 'machine learning', 'Science', 'SQL']
Total Skills: 5


In [4]:
job_descriptions = [
    {
        "job_id": "JOB001",
        "job_title": "Data Scientist",
        "company": "ABC Technologies",
        "location": "Bangalore",
        "description": """
        We are looking for a Data Scientist with strong knowledge
        of Python, SQL, Machine Learning, Deep Learning, NLP,
        Pandas, NumPy and Scikit-learn.

        Candidates should have experience in data analysis,
        predictive modelling and data visualization.

        Bachelor's degree in Computer Science or related field.
        Minimum 2 years of experience preferred.
        """
    },
    {
        "job_id": "JOB002",
        "job_title": "Machine Learning Engineer",
        "company": "XYZ Solutions",
        "location": "Hyderabad",
        "description": """
        Looking for a Machine Learning Engineer skilled in Python,
        TensorFlow, PyTorch, NLP, Deep Learning and model deployment.

        Experience with FastAPI, Docker and cloud technologies
        is preferred.
        """
    },
    {
        "job_id": "JOB003",
        "job_title": "Data Analyst",
        "company": "TechVision",
        "location": "Trivandrum",
        "description": """
        Seeking a Data Analyst with skills in SQL, Excel,
        Python, Power BI, Tableau and data visualization.

        Responsibilities include reporting, data cleaning,
        dashboard development and business analysis.
        """
    }
]

print("Job descriptions created:", len(job_descriptions))

Job descriptions created: 3


In [5]:
jobs_df = pd.DataFrame(job_descriptions)

display(jobs_df[[
    "job_id",
    "job_title",
    "company",
    "location"
]])

,job_id,job_title,company,location
0,JOB001,Data Scientist,ABC Technologies,Bangalore
1,JOB002,Machine Learning Engineer,XYZ Solutions,Hyderabad
2,JOB003,Data Analyst,TechVision,Trivandrum


In [6]:
def add_job_description(job_id, title, company, location, description):

    new_job = {
        "job_id": job_id,
        "job_title": title,
        "company": company,
        "location": location,
        "description": description
    }

    return pd.concat(
        [jobs_df, pd.DataFrame([new_job])],
        ignore_index=True
    )

In [7]:
new_job_description = """
Hiring a Python Developer with Python, Django, REST API,
MySQL, Git and problem-solving skills.
Experience in backend development is preferred.
"""

jobs_df = add_job_description(
    "JOB004",
    "Python Developer",
    "NextGen Technologies",
    "Kochi",
    new_job_description
)

print("Updated job count:", len(jobs_df))

Updated job count: 4


In [8]:
def clean_job_description(text):

    text = str(text).lower()

    # Remove HTML tags
    text = re.sub(r"<.*?>", " ", text)

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", " ", text)

    # Remove special characters but preserve technical symbols
    text = re.sub(r"[^a-z0-9+#.\s]", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [9]:
jobs_df["cleaned_description"] = (
    jobs_df["description"].apply(clean_job_description)
)

display(jobs_df[[
    "job_title",
    "cleaned_description"
]])

,job_title,cleaned_description
0,Data Scientist,we are looking for a data scientist with stron...
1,Machine Learning Engineer,looking for a machine learning engineer skille...
2,Data Analyst,seeking a data analyst with skills in sql exce...
3,Python Developer,hiring a python developer with python django r...


In [10]:
for _, row in jobs_df.iterrows():

    print("\nJob:", row["job_title"])
    print("Original length:", len(row["description"]))
    print("Cleaned length:", len(row["cleaned_description"]))
    print("Cleaned text:", row["cleaned_description"][:200])


Job: Data Scientist
Original length: 405
Cleaned length: 330
Cleaned text: we are looking for a data scientist with strong knowledge of python sql machine learning deep learning nlp pandas numpy and scikit learn. candidates should have experience in data analysis predictive 

Job: Machine Learning Engineer
Original length: 232
Cleaned length: 184
Cleaned text: looking for a machine learning engineer skilled in python tensorflow pytorch nlp deep learning and model deployment. experience with fastapi docker and cloud technologies is preferred.

Job: Data Analyst
Original length: 238
Cleaned length: 189
Cleaned text: seeking a data analyst with skills in sql excel python power bi tableau and data visualization. responsibilities include reporting data cleaning dashboard development and business analysis.

Job: Python Developer
Original length: 145
Cleaned length: 139
Cleaned text: hiring a python developer with python django rest api mysql git and problem solving skills. experience in bac

In [11]:
cleaned_jobs_path = JOB_OUTPUT / "cleaned_job_descriptions.csv"

jobs_df.to_csv(cleaned_jobs_path, index=False)

print("Cleaned job descriptions saved!")
print("Location:", cleaned_jobs_path)

Cleaned job descriptions saved!
Location: /Users/rizwansalim/Desktop/SkillBridgeByZequa/data/processed/job_analyzer/cleaned_job_descriptions.csv


In [12]:
# Load the project's skill vocabulary when it is available.
# If the CSV is missing, build a small vocabulary from the example job descriptions and candidate skills.

SKILLS_PATH = PROJECT_ROOT / "data" / "processed" / "unified_skills.csv"

if SKILLS_PATH.is_file():
    skills_df = pd.read_csv(SKILLS_PATH)
    print("Available columns:", skills_df.columns.tolist())
    print("Total skills in vocabulary:", len(skills_df))
    display(skills_df.head())
else:
    warnings.warn(
        "unified_skills.csv was not found. Using skills mentioned in the job descriptions and candidate profile."
    )
    skills_df = pd.DataFrame({"Skill Name": list(candidate_skills)})


Available columns: ['Skill ID', 'Skill Name', 'Source']
Total skills in vocabulary: 13949


,Skill ID,Skill Name,Source
0,http://data.europa.eu/esco/skill/0005c151-5b5a...,manage musical staff,ESCO
1,http://data.europa.eu/esco/skill/00064735-8fad...,supervise correctional procedures,ESCO
2,http://data.europa.eu/esco/skill/000709ed-2be5...,apply anti-oppressive practices,ESCO
3,http://data.europa.eu/esco/skill/0007bdc2-dd15...,control compliance of railway vehicles regulat...,ESCO
4,http://data.europa.eu/esco/skill/00090cc1-1f27...,identify available services,ESCO


In [13]:
# The skills CSV normally contains a column named 'Skill Name'.
# This fallback supports CSV files that use a slightly different column heading.

if "Skill Name" in skills_df.columns:
    skill_column = "Skill Name"
elif "skill_name" in skills_df.columns:
    skill_column = "skill_name"
else:
    skill_column = skills_df.columns[0]
    warnings.warn(f"Using '{skill_column}' as the skill-name column.")

skill_names = (
    skills_df[skill_column]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
    .tolist()
)

# Include candidate skills so that the candidate's vocabulary is also recognized.
skill_names = sorted(set(skill_names + [str(skill).strip() for skill in candidate_skills if str(skill).strip()]), key=str.lower)

def extract_required_skills(text, skill_list):
    text = str(text).lower()
    found_skills = []

    for skill in skill_list:
        skill_lower = str(skill).lower().strip()
        if not skill_lower:
            continue

        pattern = r"(?<![a-z0-9])" + re.escape(skill_lower) + r"(?![a-z0-9])"
        if re.search(pattern, text):
            found_skills.append(str(skill).strip())

    return sorted(set(found_skills), key=str.lower)


In [14]:
jobs_df["required_skills"] = jobs_df[
    "cleaned_description"
].apply(
    lambda text: extract_required_skills(
        text, skill_names
    )
)

for _, row in jobs_df.iterrows():

    print("\nJob:", row["job_title"])
    print("Required skills:", row["required_skills"])
    print("Skill count:", len(row["required_skills"]))


Job: Data Scientist
Required skills: ['computer science', 'deep learning', 'machine learning', 'Science', 'SQL']
Skill count: 5

Job: Machine Learning Engineer
Required skills: ['cloud technologies', 'deep learning', 'machine learning']
Skill count: 3

Job: Data Analyst
Required skills: ['business analysis', 'SQL']
Skill count: 2

Job: Python Developer
Required skills: ['MySQL']
Skill count: 1


In [15]:
jobs_df.to_csv(
    JOB_OUTPUT / "jobs_with_required_skills.csv",
    index=False
)

print("Required skills extracted and saved!")

Required skills extracted and saved!


In [16]:
def extract_qualifications(text):

    text = str(text)

    degree_patterns = [
        r"\bB\.?Tech\b",
        r"\bM\.?Tech\b",
        r"\bB\.?E\b",
        r"\bM\.?E\b",
        r"\bBachelor'?s degree\b",
        r"\bMaster'?s degree\b",
        r"\bPh\.?D\b",
        r"\bMBA\b",
        r"\bComputer Science\b",
        r"\bEngineering\b"
    ]

    found = []

    for pattern in degree_patterns:

        matches = re.findall(
            pattern, text, flags=re.IGNORECASE
        )

        found.extend(matches)

    return sorted(set(found), key=str.lower)

In [17]:
def extract_experience(text):

    patterns = [
        r"(\d+)\+?\s*(?:years|year|yrs|yr)\s+of\s+experience",
        r"minimum\s+(\d+)\s+years",
        r"at\s+least\s+(\d+)\s+years"
    ]

    results = []

    for pattern in patterns:

        matches = re.findall(
            pattern, str(text), flags=re.IGNORECASE
        )

        results.extend(matches)

    return [int(value) for value in results]

In [18]:
jobs_df["required_qualifications"] = (
    jobs_df["description"].apply(extract_qualifications)
)

jobs_df["required_experience_years"] = (
    jobs_df["description"].apply(extract_experience)
)

display(jobs_df[[
    "job_title",
    "required_qualifications",
    "required_experience_years"
]])

,job_title,required_qualifications,required_experience_years
0,Data Scientist,"[Bachelor's degree, Computer Science]","[2, 2]"
1,Machine Learning Engineer,[],[]
2,Data Analyst,[],[]
3,Python Developer,[],[]


In [19]:
jobs_df.to_csv(
    JOB_OUTPUT / "jobs_with_requirements.csv",
    index=False
)

print("Qualifications and experience extracted!")

Qualifications and experience extracted!


In [20]:
candidate_skills_clean = {
    str(skill).strip().lower()
    for skill in candidate_skills
    if str(skill).strip()
}

print("Candidate skills:")
print(candidate_skills_clean)

Candidate skills:
{'sql', 'computer science', 'deep learning', 'machine learning', 'science'}


In [21]:
def match_candidate_skills(candidate_skills, required_skills):

    candidate_set = {
        str(skill).strip().lower()
        for skill in candidate_skills
    }

    required_set = {
        str(skill).strip().lower()
        for skill in required_skills
    }

    matched = sorted(candidate_set & required_set)

    missing = sorted(required_set - candidate_set)

    return {
        "matched_skills": matched,
        "missing_skills": missing,
        "matched_count": len(matched),
        "required_count": len(required_set)
    }

In [22]:
job_match_results = []

for _, job in jobs_df.iterrows():

    result = match_candidate_skills(
        candidate_skills,
        job["required_skills"]
    )

    job_match_results.append({
        "job_id": job["job_id"],
        "job_title": job["job_title"],
        **result
    })

job_matches_df = pd.DataFrame(job_match_results)

display(job_matches_df)

,job_id,job_title,matched_skills,missing_skills,matched_count,required_count
0,JOB001,Data Scientist,"[computer science, deep learning, machine lear...",[],5,5
1,JOB002,Machine Learning Engineer,"[deep learning, machine learning]",[cloud technologies],2,3
2,JOB003,Data Analyst,[sql],[business analysis],1,2
3,JOB004,Python Developer,[],[mysql],0,1


In [23]:
job_matches_path = JOB_OUTPUT / "candidate_job_skill_matches.csv"

job_matches_df.to_csv(job_matches_path, index=False)

print("Candidate-job skill matching completed!")
print("Saved:", job_matches_path)

Candidate-job skill matching completed!
Saved: /Users/rizwansalim/Desktop/SkillBridgeByZequa/data/processed/job_analyzer/candidate_job_skill_matches.csv


In [24]:
# Semantic matching can find related terms, but it needs sentence-transformers and a model download.
# Keep it disabled by default so the notebook runs without a large model installation.

semantic_model = None

if ENABLE_SEMANTIC_MATCHING:
    try:
        import os
        os.environ["USE_TF"] = "0"
        os.environ["USE_FLAX"] = "0"
        from sentence_transformers import SentenceTransformer

        semantic_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
        print("Semantic model loaded successfully!")
    except ImportError as error:
        warnings.warn(
            "Semantic matching was requested but sentence-transformers is not installed. "
            "Run %pip install sentence-transformers, then restart the kernel. "
            f"Details: {error}"
        )
        ENABLE_SEMANTIC_MATCHING = False
    except Exception as error:
        warnings.warn(f"Semantic model could not be loaded. Exact matching will continue. Details: {error}")
        ENABLE_SEMANTIC_MATCHING = False
else:
    print("Semantic matching is disabled. Exact skill matching will be used.")


Semantic matching is disabled. Exact skill matching will be used.


In [25]:
def semantic_skill_matching(candidate_skills, required_skills, threshold=0.65):
    """Find related skills using sentence embeddings when the optional model is enabled."""

    if not ENABLE_SEMANTIC_MATCHING or semantic_model is None:
        return []

    if not candidate_skills or not required_skills:
        return []

    candidate_embeddings = semantic_model.encode(
        candidate_skills,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    required_embeddings = semantic_model.encode(
        required_skills,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    similarity_matrix = candidate_embeddings @ required_embeddings.T
    matches = []

    for required_index, required_skill in enumerate(required_skills):
        best_candidate_index = int(np.argmax(similarity_matrix[:, required_index]))
        similarity = float(similarity_matrix[best_candidate_index, required_index])

        if similarity >= threshold:
            matches.append({
                "candidate_skill": candidate_skills[best_candidate_index],
                "required_skill": required_skill,
                "similarity": round(similarity, 4)
            })

    return matches


In [26]:
semantic_job_results = []

for _, job in jobs_df.iterrows():

    matches = semantic_skill_matching(
        candidate_skills,
        job["required_skills"],
        threshold=0.65
    )

    for match in matches:
        semantic_job_results.append({
            "job_id": job["job_id"],
            "job_title": job["job_title"],
            **match
        })

semantic_matches_df = pd.DataFrame(
    semantic_job_results
)

display(semantic_matches_df)

""


In [27]:
semantic_path = (
    JOB_OUTPUT / "semantic_job_skill_matches.csv"
)

semantic_matches_df.to_csv(
    semantic_path,
    index=False
)

print("Semantic job matching completed!")
print("Total semantic matches:", len(semantic_matches_df))
print("Saved:", semantic_path)

Semantic job matching completed!
Total semantic matches: 0
Saved: /Users/rizwansalim/Desktop/SkillBridgeByZequa/data/processed/job_analyzer/semantic_job_skill_matches.csv


In [28]:
def calculate_job_match_score(
    candidate_skills,
    required_skills,
    semantic_matches
):

    required_set = {
        str(skill).strip().lower()
        for skill in required_skills
        if str(skill).strip()
    }

    if not required_set:
        return {
            "exact_coverage": 0,
            "semantic_coverage": 0,
            "job_match_score": 0
        }

    exact_matches = match_candidate_skills(
        candidate_skills,
        required_skills
    )["matched_skills"]

    exact_coverage = (
        len(exact_matches) / len(required_set)
    ) * 100

    semantic_coverage = (
        len(semantic_matches) / len(required_set)
    ) * 100

    semantic_coverage = min(semantic_coverage, 100)

    final_score = (
        exact_coverage * 0.70
        + semantic_coverage * 0.30
    )

    return {
        "exact_coverage": round(exact_coverage, 2),
        "semantic_coverage": round(semantic_coverage, 2),
        "job_match_score": round(final_score, 2)
    }

In [29]:
job_score_results = []

for _, job in jobs_df.iterrows():

    semantic_matches = semantic_skill_matching(
        candidate_skills,
        job["required_skills"],
        threshold=0.65
    )

    scores = calculate_job_match_score(
        candidate_skills,
        job["required_skills"],
        semantic_matches
    )

    job_score_results.append({
        "job_id": job["job_id"],
        "job_title": job["job_title"],
        "company": job["company"],
        **scores
    })

job_scores_df = pd.DataFrame(job_score_results)

display(
    job_scores_df.sort_values(
        "job_match_score",
        ascending=False
    )
)



,job_id,job_title,company,exact_coverage,semantic_coverage,job_match_score
0,JOB001,Data Scientist,ABC Technologies,100.00,0.0,70.00
1,JOB002,Machine Learning Engineer,XYZ Solutions,66.67,0.0,46.67
2,JOB003,Data Analyst,TechVision,50.00,0.0,35.00
3,JOB004,Python Developer,NextGen Technologies,0.00,0.0,0.00


In [30]:
score_path = JOB_OUTPUT / "job_match_scores.csv"

job_scores_df.to_csv(score_path, index=False)

print("Job match scores calculated!")
print("Saved:", score_path)

Job match scores calculated!
Saved: /Users/rizwansalim/Desktop/SkillBridgeByZequa/data/processed/job_analyzer/job_match_scores.csv


In [31]:
def analyze_missing_skills(
    candidate_skills,
    required_skills,
    semantic_matches
):

    exact_matched = {
        skill.lower()
        for skill in match_candidate_skills(
            candidate_skills,
            required_skills
        )["matched_skills"]
    }

    semantic_matched = {
        match["required_skill"].lower()
        for match in semantic_matches
    }

    missing_skills = [
        skill for skill in required_skills
        if skill.lower() not in exact_matched
        and skill.lower() not in semantic_matched
    ]

    related_skills = [
        match for match in semantic_matches
        if match["required_skill"].lower() not in exact_matched
    ]

    return {
        "confirmed_matched_skills": sorted(exact_matched),
        "suggested_related_skills": related_skills,
        "missing_skills": sorted(missing_skills)
    }

In [32]:
skill_gap_results = []

for _, job in jobs_df.iterrows():

    semantic_matches = semantic_skill_matching(
        candidate_skills,
        job["required_skills"],
        threshold=0.65
    )

    gap = analyze_missing_skills(
        candidate_skills,
        job["required_skills"],
        semantic_matches
    )

    skill_gap_results.append({
        "job_id": job["job_id"],
        "job_title": job["job_title"],
        **gap
    })

skill_gap_df = pd.DataFrame(skill_gap_results)

display(skill_gap_df)

,job_id,job_title,confirmed_matched_skills,suggested_related_skills,missing_skills
0,JOB001,Data Scientist,"[computer science, deep learning, machine lear...",[],[]
1,JOB002,Machine Learning Engineer,"[deep learning, machine learning]",[],[cloud technologies]
2,JOB003,Data Analyst,[sql],[],[business analysis]
3,JOB004,Python Developer,[],[],[MySQL]


In [33]:
skill_gap_path = JOB_OUTPUT / "job_skill_gap_analysis.csv"

skill_gap_df.to_csv(
    skill_gap_path,
    index=False
)

print("Skill gap analysis saved!")
print("Location:", skill_gap_path)

Skill gap analysis saved!
Location: /Users/rizwansalim/Desktop/SkillBridgeByZequa/data/processed/job_analyzer/job_skill_gap_analysis.csv


In [34]:
def rank_job_roles(job_scores_df):

    ranked_jobs = job_scores_df.copy()

    ranked_jobs = ranked_jobs.sort_values(
        by="job_match_score",
        ascending=False
    ).reset_index(drop=True)

    ranked_jobs["rank"] = (
        ranked_jobs.index + 1
    )

    return ranked_jobs

In [35]:
ranked_jobs_df = rank_job_roles(job_scores_df)

display(
    ranked_jobs_df[
        [
            "rank",
            "job_title",
            "company",
            "job_match_score",
            "exact_coverage",
            "semantic_coverage"
        ]
    ]
)

,rank,job_title,company,job_match_score,exact_coverage,semantic_coverage
0,1,Data Scientist,ABC Technologies,70.00,100.00,0.0
1,2,Machine Learning Engineer,XYZ Solutions,46.67,66.67,0.0
2,3,Data Analyst,TechVision,35.00,50.00,0.0
3,4,Python Developer,NextGen Technologies,0.00,0.00,0.0


In [36]:
top_3_jobs = ranked_jobs_df.head(3)

print("TOP 3 JOB RECOMMENDATIONS")
print("-" * 35)

for _, job in top_3_jobs.iterrows():

    print(f"\nRank: {job['rank']}")
    print(f"Role: {job['job_title']}")
    print(f"Company: {job['company']}")
    print(f"Match Score: {job['job_match_score']}%")

TOP 3 JOB RECOMMENDATIONS
-----------------------------------

Rank: 1
Role: Data Scientist
Company: ABC Technologies
Match Score: 70.0%

Rank: 2
Role: Machine Learning Engineer
Company: XYZ Solutions
Match Score: 46.67%

Rank: 3
Role: Data Analyst
Company: TechVision
Match Score: 35.0%


In [37]:
ranking_path = JOB_OUTPUT / "ranked_job_recommendations.csv"

ranked_jobs_df.to_csv(
    ranking_path,
    index=False
)

print("Job rankings saved successfully!")
print("Location:", ranking_path)

Job rankings saved successfully!
Location: /Users/rizwansalim/Desktop/SkillBridgeByZequa/data/processed/job_analyzer/ranked_job_recommendations.csv


In [38]:
def generate_job_analysis_report(
    candidate_data,
    jobs_df,
    ranked_jobs_df,
    skill_gap_df
):

    candidate_details = candidate_data.get(
        "personal_details", {}
    )

    report = {
        "candidate": {
            "name": candidate_details.get("full_name") or candidate_data.get("name", "Candidate"),
            "email": candidate_details.get("email", ""),
            "skills": candidate_data.get("skills", [])
        },
        "total_jobs_analyzed": len(jobs_df),
        "job_recommendations": [],
        "analysis_summary": {}
    }

    for _, job in ranked_jobs_df.iterrows():

        job_id = job["job_id"]

        gap_row = skill_gap_df[
            skill_gap_df["job_id"] == job_id
        ]

        if gap_row.empty:
            continue

        gap = gap_row.iloc[0]

        report["job_recommendations"].append({
            "rank": int(job["rank"]),
            "job_id": job_id,
            "job_title": job["job_title"],
            "company": job["company"],
            "match_score": float(job["job_match_score"]),
            "exact_coverage": float(job["exact_coverage"]),
            "semantic_coverage": float(job["semantic_coverage"]),
            "missing_skills": gap["missing_skills"],
            "suggested_related_skills": gap[
                "suggested_related_skills"
            ]
        })

    if len(ranked_jobs_df) > 0:

        best_job = ranked_jobs_df.iloc[0]

        report["analysis_summary"] = {
            "best_matching_role": best_job["job_title"],
            "best_matching_company": best_job["company"],
            "best_match_score": float(
                best_job["job_match_score"]
            )
        }

    return report

In [39]:
job_analysis_report = generate_job_analysis_report(
    candidate_data,
    jobs_df,
    ranked_jobs_df,
    skill_gap_df
)

print(json.dumps(
    job_analysis_report,
    indent=4
))

{
    "candidate": {
        "name": "Candidate",
        "email": "",
        "skills": [
            "computer science",
            "deep learning",
            "machine learning",
            "Science",
            "SQL"
        ]
    },
    "total_jobs_analyzed": 4,
    "job_recommendations": [
        {
            "rank": 1,
            "job_id": "JOB001",
            "job_title": "Data Scientist",
            "company": "ABC Technologies",
            "match_score": 70.0,
            "exact_coverage": 100.0,
            "semantic_coverage": 0.0,
            "missing_skills": [],
            "suggested_related_skills": []
        },
        {
            "rank": 2,
            "job_id": "JOB002",
            "job_title": "Machine Learning Engineer",
            "company": "XYZ Solutions",
            "match_score": 46.67,
            "exact_coverage": 66.67,
            "semantic_coverage": 0.0,
            "missing_skills": [
                "cloud technologies"
            ],


In [40]:
print("CANDIDATE:", job_analysis_report["candidate"]["name"])

print(
    "\nTOTAL JOBS ANALYZED:",
    job_analysis_report["total_jobs_analyzed"]
)

print("\nJOB RECOMMENDATIONS")

for job in job_analysis_report["job_recommendations"]:

    print("\nRank:", job["rank"])
    print("Role:", job["job_title"])
    print("Company:", job["company"])
    print("Match:", job["match_score"], "%")
    print("Missing skills:", job["missing_skills"])

CANDIDATE: Candidate

TOTAL JOBS ANALYZED: 4

JOB RECOMMENDATIONS

Rank: 1
Role: Data Scientist
Company: ABC Technologies
Match: 70.0 %
Missing skills: []

Rank: 2
Role: Machine Learning Engineer
Company: XYZ Solutions
Match: 46.67 %
Missing skills: ['cloud technologies']

Rank: 3
Role: Data Analyst
Company: TechVision
Match: 35.0 %
Missing skills: ['business analysis']

Rank: 4
Role: Python Developer
Company: NextGen Technologies
Match: 0.0 %
Missing skills: ['MySQL']


In [41]:
JOB_OUTPUT.mkdir(parents=True, exist_ok=True)

print("Output folder ready:", JOB_OUTPUT)

Output folder ready: /Users/rizwansalim/Desktop/SkillBridgeByZequa/data/processed/job_analyzer


In [42]:
report_path = JOB_OUTPUT / "complete_job_analysis_report.json"

with open(report_path, "w", encoding="utf-8") as file:
    json.dump(
        job_analysis_report,
        file,
        indent=4,
        ensure_ascii=False
    )

print("JSON report saved:", report_path)

JSON report saved: /Users/rizwansalim/Desktop/SkillBridgeByZequa/data/processed/job_analyzer/complete_job_analysis_report.json


In [43]:
output_files = {
    "cleaned_jobs": jobs_df,
    "skill_matches": job_matches_df,
    "semantic_matches": semantic_matches_df,
    "job_scores": job_scores_df,
    "skill_gaps": skill_gap_df,
    "ranked_jobs": ranked_jobs_df
}

for name, dataframe in output_files.items():

    output_path = JOB_OUTPUT / f"{name}.csv"

    dataframe.to_csv(
        output_path,
        index=False
    )

    print("Saved:", output_path.name)

Saved: cleaned_jobs.csv
Saved: skill_matches.csv
Saved: semantic_matches.csv
Saved: job_scores.csv
Saved: skill_gaps.csv
Saved: ranked_jobs.csv


In [44]:
print("\nDAY 10 OUTPUT FILES")
print("-" * 40)

for file in sorted(JOB_OUTPUT.iterdir()):

    if file.is_file():
        print(
            f"{file.name} | "
            f"{file.stat().st_size:,} bytes"
        )


DAY 10 OUTPUT FILES
----------------------------------------
candidate_job_skill_matches.csv | 378 bytes
cleaned_job_descriptions.csv | 2,143 bytes
cleaned_jobs.csv | 2,463 bytes
complete_job_analysis_report.json | 2,001 bytes
job_match_scores.csv | 293 bytes
job_scores.csv | 293 bytes
job_skill_gap_analysis.csv | 380 bytes
jobs_with_required_skills.csv | 2,340 bytes
jobs_with_requirements.csv | 2,463 bytes
ranked_job_recommendations.csv | 306 bytes
ranked_jobs.csv | 306 bytes
semantic_job_skill_matches.csv | 1 bytes
semantic_matches.csv | 1 bytes
skill_gaps.csv | 380 bytes
skill_matches.csv | 378 bytes


In [45]:
required_files = [
    "cleaned_jobs.csv",
    "skill_matches.csv",
    "semantic_matches.csv",
    "job_scores.csv",
    "skill_gaps.csv",
    "ranked_jobs.csv",
    "complete_job_analysis_report.json"
]

print("SKILLBRIDGE DAY 10 VALIDATION")
print("-" * 45)

all_files_found = True
for filename in required_files:
    file_path = JOB_OUTPUT / filename
    exists = file_path.is_file()
    print(("FOUND " if exists else "MISSING"), "|", filename)
    if not exists:
        all_files_found = False

print("-" * 45)

report_file = JOB_OUTPUT / "complete_job_analysis_report.json"
if report_file.is_file():
    with report_file.open("r", encoding="utf-8") as file:
        saved_report = json.load(file)

    print("Candidate:", saved_report.get("candidate", {}).get("name", "Candidate"))
    print("Jobs analyzed:", saved_report.get("total_jobs_analyzed", 0))
    print("Recommendations:", len(saved_report.get("job_recommendations", [])))
    print("Best role:", saved_report.get("analysis_summary", {}).get("best_matching_role", "Not available"))
else:
    saved_report = {}

print("-" * 45)
if all_files_found and saved_report.get("job_recommendations"):
    print("VALIDATION PASSED")
else:
    print("VALIDATION NEEDS ATTENTION")


SKILLBRIDGE DAY 10 VALIDATION
---------------------------------------------
FOUND  | cleaned_jobs.csv
FOUND  | skill_matches.csv
FOUND  | semantic_matches.csv
FOUND  | job_scores.csv
FOUND  | skill_gaps.csv
FOUND  | ranked_jobs.csv
FOUND  | complete_job_analysis_report.json
---------------------------------------------
Candidate: Candidate
Jobs analyzed: 4
Recommendations: 4
Best role: Data Scientist
---------------------------------------------
VALIDATION PASSED
